# Notebook 16 — Canonical CZ Extraction, Local Z Compensation, and Error Budget

This notebook takes the best shaped-adiabatic gate candidate and pushes it one step closer to a paper-style characterization.

It does four things:

1. builds the best extracted shaped gate,
2. estimates and removes local single-qubit diagonal phases,
3. compares the compensated gate against canonical CZ,
4. reports a compact error budget:
   - raw process fidelity,
   - phase-corrected fidelity,
   - compensated CZ fidelity,
   - entangling phase,
   - leakage,
   - matrix errors.

This is the natural next step after Notebook 15b because a high-quality Rydberg controlled-phase gate is usually interpreted **up to local Z corrections**.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import basis, qeye, tensor, sigmax, sesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)

    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)

    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Effective-unitary helpers

In [ ]:
def run_shaped_evolution(psi0, T, omega_max, alpha, V, delta0, p, q, n_steps=500):
    H = build_time_dependent_hamiltonian(
        T=T, omega_max=omega_max, alpha=alpha, V=V, delta0=delta0, p=p, q=q
    )
    times = np.linspace(0.0, T, n_steps)
    result = sesolve(H, psi0, times)
    return result.states[-1]

def state_to_column(psi):
    return np.array([basis_state.overlap(psi) for basis_state in basis_states], dtype=complex)

def build_effective_unitary(T, omega_max, alpha, V, delta0, p, q, n_steps=500):
    columns = []
    for psi0 in basis_states:
        psi_final = run_shaped_evolution(
            psi0, T=T, omega_max=omega_max, alpha=alpha, V=V,
            delta0=delta0, p=p, q=q, n_steps=n_steps
        )
        columns.append(state_to_column(psi_final))
    return np.column_stack(columns)


## Gate metrics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def phase_corrected_target(U_eff):
    phi_ent = entangling_phase(U_eff)
    return np.diag([1, 1, 1, np.exp(1j * phi_ent)]).astype(complex)

def phase_corrected_fidelity(U_eff):
    return process_fidelity(U_eff, phase_corrected_target(U_eff))

def leakage_metric(U_eff):
    offdiag = U_eff.copy()
    np.fill_diagonal(offdiag, 0)
    return float(np.linalg.norm(offdiag))

def matrix_error(U_eff, U_target):
    return float(np.linalg.norm(U_eff - U_target))

def composite_score(U_eff):
    return (
        0.5 * phase_corrected_fidelity(U_eff)
        + 0.3 * process_fidelity(U_eff, U_cz)
        - 0.2 * leakage_metric(U_eff)
    )


## Best extracted shaped candidate from Notebook 15b

In [ ]:
best = {
    'T': 26.0,
    'alpha': 0.18,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

U_best = build_effective_unitary(
    T=best['T'],
    omega_max=best['omega_max'],
    alpha=best['alpha'],
    V=V,
    delta0=best['delta0'],
    p=best['p'],
    q=best['q'],
    n_steps=500,
)

print("Best extracted candidate parameters:")
for k, v in best.items():
    print(f"{k}: {v:.4f}")


## Local Z-phase extraction

In [ ]:
def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

global_phase, a, b, phi_ent = extract_local_z_phases(U_best)
print(f"global phase = {global_phase:.6f}")
print(f"a (qubit 1 phase) = {a:.6f}")
print(f"b (qubit 2 phase) = {b:.6f}")
print(f"entangling phase = {phi_ent:.6f} rad = {phi_ent/np.pi:.6f} pi")


## Canonical CZ compensation

In [ ]:
def compensate_to_cz(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    L = local_z_diagonal(-a, -b)
    U2 = U1 @ L
    cp_target = np.diag([1, 1, 1, np.exp(1j * phi_ent)]).astype(complex)
    return U2, cp_target

U_comp, U_cp = compensate_to_cz(U_best)
U_canonical_target = U_cz.copy()

raw_fid = process_fidelity(U_best, U_cz)
pc_fid = phase_corrected_fidelity(U_best)
comp_cp_fid = process_fidelity(U_comp, U_cp)
comp_cz_fid = process_fidelity(U_comp, U_canonical_target)

print(f"Raw fidelity vs CZ: {raw_fid:.6f}")
print(f"Phase-corrected fidelity: {pc_fid:.6f}")
print(f"Compensated fidelity vs extracted CP target: {comp_cp_fid:.6f}")
print(f"Compensated fidelity vs canonical CZ: {comp_cz_fid:.6f}")


## Error budget

In [ ]:
error_budget = {
    'raw_fidelity_vs_CZ': raw_fid,
    'phase_corrected_fidelity': pc_fid,
    'compensated_fidelity_vs_CP_target': comp_cp_fid,
    'compensated_fidelity_vs_CZ': comp_cz_fid,
    'entangling_phase_over_pi': phi_ent / np.pi,
    'leakage': leakage_metric(U_best),
    'matrix_error_vs_CZ': matrix_error(U_best, U_cz),
    'matrix_error_vs_phase_corrected_target': matrix_error(U_best, phase_corrected_target(U_best)),
    'matrix_error_after_compensation_vs_CP_target': matrix_error(U_comp, U_cp),
    'composite_score': composite_score(U_best),
}

for k, v in error_budget.items():
    print(f"{k}: {v:.6f}")


## Figure 1 — best unitary, compensated unitary, and ideal CZ

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))

im0 = axes[0].imshow(np.abs(U_best), origin='lower', aspect='equal')
axes[0].set_title(r'$|U_{best}|$')
axes[0].set_xticks(range(4), basis_labels)
axes[0].set_yticks(range(4), basis_labels)

im1 = axes[1].imshow(np.abs(U_comp), origin='lower', aspect='equal')
axes[1].set_title(r'$|U_{comp}|$')
axes[1].set_xticks(range(4), basis_labels)
axes[1].set_yticks(range(4), basis_labels)

im2 = axes[2].imshow(np.abs(U_cz), origin='lower', aspect='equal')
axes[2].set_title(r'$|U_{CZ}|$')
axes[2].set_xticks(range(4), basis_labels)
axes[2].set_yticks(range(4), basis_labels)

fig.colorbar(im1, ax=axes.ravel().tolist(), label='magnitude')
plt.tight_layout()
plt.show()


## Figure 2 — diagonal phases before and after compensation

In [ ]:
ph_best = diagonal_phases(U_best) / np.pi
ph_comp = diagonal_phases(U_comp) / np.pi

x = np.arange(4)
w = 0.35

plt.figure(figsize=(7, 4.5))
plt.bar(x - w/2, ph_best, width=w, label='before compensation')
plt.bar(x + w/2, ph_comp, width=w, label='after compensation')
plt.xticks(x, basis_labels)
plt.ylabel(r'Diagonal phase / $\pi$')
plt.title('Diagonal phases before and after local Z compensation')
plt.legend()
plt.tight_layout()
plt.show()


## Figure 3 — compact error budget

In [ ]:
labels = [
    'raw F',
    'phase F',
    'comp CP F',
    'comp CZ F',
]
values = [
    error_budget['raw_fidelity_vs_CZ'],
    error_budget['phase_corrected_fidelity'],
    error_budget['compensated_fidelity_vs_CP_target'],
    error_budget['compensated_fidelity_vs_CZ'],
]

plt.figure(figsize=(7, 4.5))
plt.bar(labels, values)
plt.ylim(0, 1.05)
plt.ylabel('Fidelity')
plt.title('Canonical CZ extraction fidelity ladder')
plt.tight_layout()
plt.show()

labels2 = ['leakage', '||U-CZ||', '||U-CP||', '||Ucomp-CP||']
values2 = [
    error_budget['leakage'],
    error_budget['matrix_error_vs_CZ'],
    error_budget['matrix_error_vs_phase_corrected_target'],
    error_budget['matrix_error_after_compensation_vs_CP_target'],
]
plt.figure(figsize=(7, 4.5))
plt.bar(labels2, values2)
plt.ylabel('Error magnitude')
plt.title('Error budget')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## Summary block

In [ ]:
summary_text = f'''
Canonical CZ extraction summary

Best shaped candidate:
T      = {best["T"]:.4f}
alpha  = {best["alpha"]:.4f}
Omega  = {best["omega_max"]:.4f}
Delta0 = {best["delta0"]:.4f}
p      = {best["p"]:.4f}
q      = {best["q"]:.4f}

Extracted phase structure:
global phase   = {global_phase:.4f}
local a        = {a:.4f}
local b        = {b:.4f}
phi_ent / pi   = {phi_ent/np.pi:.4f}

Fidelity ladder:
raw F vs CZ                = {raw_fid:.4f}
phase-corrected F          = {pc_fid:.4f}
compensated F vs CP target = {comp_cp_fid:.4f}
compensated F vs CZ        = {comp_cz_fid:.4f}

Other diagnostics:
leakage                    = {error_budget["leakage"]:.4f}
||U - CZ||                 = {error_budget["matrix_error_vs_CZ"]:.4f}
||U - phase target||       = {error_budget["matrix_error_vs_phase_corrected_target"]:.4f}
||Ucomp - CP target||      = {error_budget["matrix_error_after_compensation_vs_CP_target"]:.4f}
composite score            = {error_budget["composite_score"]:.4f}
'''
print(summary_text)


## Final conclusion

This notebook shows that the best shaped-adiabatic gate can be interpreted as a high-quality controlled-phase gate with residual local diagonal phases.

After local Z compensation, the gate can be compared much more directly to canonical CZ.

That gives a cleaner paper-style statement:

**the shaped-adiabatic protocol produces a near-diagonal, low-leakage controlled-phase gate, and its remaining discrepancy from canonical CZ is largely attributable to removable local phase structure plus a smaller residual coherent error budget.**
